In [5]:
import os
import glob
import gradio as gr
from rich import print
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

In Python, `import glob` imports a built-in module used to search for files and directories using pattern matching.

It works similarly to the wildcards you use in a command line terminal or file explorer search bar (like searching for `*.txt` to find all text files).

#### How it works

The name "glob" is short for __global__, which is an old Unix term for expanding wildcards into a list of matching pathnames.

Instead of opening a folder and manually filtering through files with loops and conditional `if` statements, `glob` lets you specify a pattern, and it returns a neat Python list containing all matching file paths.

#### Common Use Cases

Here are the most common patterns you can use with `glob`:

1. __Find all files of a specific extension__

If you want to pull a list of every single CSV file in your current working directory:

`import glob`

`# Finds all files ending with .csv`

`csv_files = glob.glob("*.csv")`

`print(csv_files) `

`# Output: ['data1.csv', 'report_final.csv', 'users.csv']`

2. __Match patterns with variables__

If you have files named `image_1.png`, `image_2.png`, and `image_backup.png`, you can use the `?` wildcard to match exactly one character:

`# The '?' matches exactly one single character (like numbers 0-9)`

`numbered_images = glob.glob("image_?.png")`

`print(numbered_images)`

`# Output: ['image_1.png', 'image_2.png'] -> skips 'image_backup.png'`

3. __Search deeply inside subfolders (Recursive Search)__

If you have a complex project folder and want to find every Python file (`.py`) buried deep inside any subfolder, you use the double-asterisk  alongside the `recursive=True` flag:

`# Looks inside the current folder and EVERY folder below it`

`all_python_files = glob.glob("**/*.py", recursive=True)`

In [74]:
# Loading environment variables from .env file

load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL")
groq_base_url = os.getenv("GROQ_BASE_URL")

if groq_api_key:
    print(f"Groq API Key exists: {groq_api_key[:4]}, {groq_model[-8:]}, {groq_base_url[-8:]}")
else:
    print("Groq API Key not set.")

Groq API Key exists: gsk_, -oss-20b, penai/v1

In [7]:
openai = OpenAI(api_key=groq_api_key, base_url=groq_base_url)

Let's read in all employees data into a dictionary.

In [30]:
# Loading knowledge base from files in the "knowledge-base/employees" directory
knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[1]  # Extracting the name from the filename
    with open(filename, 'r', encoding="utf-8") as file:
        knowledge[name.lower()] = file.read()

In [24]:
print(f"{knowledge[name.lower()][:100]}")

# HR Record

# Tyler Brooks

## Summary
- **Date of Birth:** September 9, 1998
- **Job Title:** Juni

In [29]:
print(f"{knowledge["brooks"][:100]}")

# HR Record

# Tyler Brooks

## Summary
- **Date of Birth:** September 9, 1998
- **Job Title:** Juni

In [32]:
# Loading knowledge base from files in the "knowledge-base/products" directory
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, 'r', encoding="utf-8") as file:
        knowledge[name.lower()] = file.read()

In [37]:
print(knowledge.keys())
# print(f"{knowledge['chen'][:100]}")

dict_keys(['chen', 'harper', 'thomson', 'foster', 'lancaster', 'walker', 'rodriguez', 'park', 'kim', 'carter', 
'tran', 'wilson', 'adams', 'liu', 'blake', 'k.', 'zhang', 'anderson', 'johnson', 'thompson', "o'brien", 'rivera', 
'patel', 'spencer', 'sharma', 'martinez', 'greene', 'trenton', 'williams', 'brooks', 'bizllm', 'carllm', 
'claimllm', 'healthllm', 'homellm', 'lifellm', 'markellm', 'rellm'])

In [38]:
SYSTEM_PREFIX="""
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [59]:
# A simple function to extract relevant context based on keyword matching using the pydantic way of defining a schema for the knowledge base and then using that schema to validate and extract relevant information from the user's question. This is a very basic implementation and can be improved with more sophisticated NLP techniques.
def get_relevant_context(message):
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]

In [60]:
get_relevant_context("What is the name of the CEO of Insurellm?")

[]

In [63]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n"
        result += "\n".join(relevant_context)
    return result

In [64]:
print(additional_context("Who is Alex Lancaster?"))

The following additional context might be relevant in answering the user's question:
# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a 
leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management 
expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she
developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences
in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.

## Annual Performance History
- **2015**: **Exceeds Expectations**  
  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial 
funding.  

- **2016**: **Meets Expectations**  
  Growth continued, though challenges arose in operational efficiency that required Avery's attention.  

- **2017**: **Developing**  
  Market competition intensified, and monthly sales metrics were below targets. Avery implemented new strategies 
which required a steep learning curve.  

- **2018**: **Exceeds Expectations**  
  Under Avery’s pivoted vision, Insurellm launched two new successful products that significantly increased market 
share.  

- **2019**: **Meets Expectations**  
  Steady growth, however, some team tensions led to a minor drop in employee morale. Avery recognized the need to 
enhance company culture.  

- **2020**: **Below Expectations**  
  The COVID-19 pandemic posed unforeseen operational difficulties. Avery faced criticism for delayed strategy 
shifts, although efforts were eventually made to stabilize the company.  

- **2021**: **Exceptional**  
  Avery's decisive transition to remote work and rapid adoption of digital tools led to record-high customer 
satisfaction levels and increased sales.  

- **2022**: **Satisfactory**  
  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement 
despite a saturated market.  

- **2023**: **Exceeds Expectations**  
  Market leadership was regained with innovative approaches to personalized insurance solutions. Avery is now 
recognized in industry publications as a leading voice in Insurance Tech innovation.

## Compensation History
- **2015**: $150,000 base salary + Significant equity stake  
- **2016**: $160,000 base salary + Equity increase  
- **2017**: $150,000 base salary + Decrease in bonus due to performance  
- **2018**: $180,000 base salary + performance bonus of $30,000  
- **2019**: $185,000 base salary + market adjustment + $5,000 bonus  
- **2020**: $170,000 base salary (temporary reduction due to COVID-19)  
- **2021**: $200,000 base salary + performance bonus of $50,000  
- **2022**: $210,000 base salary + retention bonus  
- **2023**: $225,000 base salary + $75,000 performance bonus  

## Other HR Notes
- **Professional Development**: Avery has actively participated in leadership training programs and industry 
conferences, representing Insurellm and fostering partnerships.  
- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing
visible improvements in team representation since 2021.  
- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by 
implementing flexible wor

In [82]:
# Creating a chat history to store the conversation between the user and the assistant
def _normalize_history(history):
    """Convert Gradio history to strict OpenAI chat messages without unsupported keys."""
    cleaned = []
    for item in history or []:
        # Gradio can provide tuple history: (user_text, assistant_text)
        if isinstance(item, (list, tuple)) and len(item) == 2:
            user_text, assistant_text = item
            if user_text:
                cleaned.append({"role": "user", "content": str(user_text)})
            if assistant_text:
                cleaned.append({"role": "assistant", "content": str(assistant_text)})
            continue

        # Gradio can provide message dicts with extra fields like metadata.
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
            if role in {"system", "user", "assistant"}:
                cleaned.append({"role": role, "content": str(content)})

    return cleaned


def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}]
    messages += _normalize_history(history)
    messages += [{"role": "user", "content": message}]

    response = openai.chat.completions.create(model=groq_model, messages=messages)
    return response.choices[0].message.content

d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await

#### Now we'll bring this up in GRADIO using the chat interface.

In [ ]:
gr.ChatInterface(fn=chat, title="Insurellm Expert Knowledge Worker", description="Ask questions about Insurellm, its employees and its products.",flagging_mode="never", show_progress="full").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await